In [76]:
import torch

In [77]:
logits = torch.randn(10)
logits

tensor([-1.0326, -0.6253, -1.2880,  0.6490, -0.5364, -0.6658, -0.9862, -0.1854,
        -0.1097,  0.4748])

In [78]:
torch.softmax(logits, dim=0)

tensor([0.0451, 0.0678, 0.0350, 0.2426, 0.0741, 0.0652, 0.0473, 0.1053, 0.1136,
        0.2038])

# top k

In [79]:
#top k
k = 5
print(torch.topk(logits, k), "\n") #this giv

smallest_value = torch.topk(logits, k).values[-1]
print(smallest_value, "\n")

top_k_values = logits.masked_fill(logits < smallest_value, float("-inf"))
print(top_k_values)
print(torch.softmax(top_k_values, dim=-1))

torch.return_types.topk(
values=tensor([ 0.6490,  0.4748, -0.1097, -0.1854, -0.5364]),
indices=tensor([3, 9, 8, 7, 4])) 

tensor(-0.5364) 

tensor([   -inf,    -inf,    -inf,  0.6490, -0.5364,    -inf,    -inf, -0.1854,
        -0.1097,  0.4748])
tensor([0.0000, 0.0000, 0.0000, 0.3281, 0.1003, 0.0000, 0.0000, 0.1424, 0.1536,
        0.2756])


# Top P

In [80]:
top_p = 0.5

In [81]:
sorted_logits, sorted_idx = torch.sort(logits, descending=True)
print(sorted_logits, sorted_idx, "\n")


tensor([ 0.6490,  0.4748, -0.1097, -0.1854, -0.5364, -0.6253, -0.6658, -0.9862,
        -1.0326, -1.2880]) tensor([3, 9, 8, 7, 4, 1, 5, 6, 0, 2]) 



In [82]:
cumulative = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
cumulative

tensor([0.2426, 0.4465, 0.5601, 0.6654, 0.7396, 0.8074, 0.8726, 0.9199, 0.9650,
        1.0000])

In [83]:
torch.softmax(sorted_logits, dim=-1)

tensor([0.2426, 0.2038, 0.1136, 0.1053, 0.0741, 0.0678, 0.0652, 0.0473, 0.0451,
        0.0350])

In [ ]:
cumulative - torch.softmax(sorted_logits, dim=-1) #cumlative - softmax because it offsets it by 1
# i.e. if the 2nd logit has a cumulative of 55% and the top p is 50% it will -inf it out when we (>top_p).
# We dont want that, we want to include the token which breaks the top_p threshold. Hence we off by 1 it via the softmax

tensor([0.0000, 0.2426, 0.4465, 0.5601, 0.6654, 0.7396, 0.8074, 0.8726, 0.9199,
        0.9650])

In [87]:
sorted_logits = sorted_logits.masked_fill(cumulative - torch.softmax(sorted_logits, dim=-1) > top_p, float("-inf"))
sorted_logits

tensor([ 0.6490,  0.4748, -0.1097,    -inf,    -inf,    -inf,    -inf,    -inf,
           -inf,    -inf])

In [ ]:
torch.empty_like(logits).scatter_

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [ ]:
logits = torch.empty_like(logits).scatter_(0, sorted_idx, sorted_logits) #scatter_ gives you index's and tells you what to write in them
logits

tensor([   -inf,    -inf,    -inf,  0.6490,    -inf,    -inf,    -inf,    -inf,
        -0.1097,  0.4748])